In [ ]:
# GPU 사용 설정
import tensorflow as tf
print("GPU 사용 가능:", tf.config.list_physical_devices('GPU'))

GPU 사용 가능: []


In [ ]:
# 필요한 라이브러리
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# 데이터셋 로드
(ds_train, ds_test), ds_info = tfds.load(
    'rock_paper_scissors',
    split=['train', 'test'],
    with_info=True,
    as_supervised=True
)

print(f"훈련 데이터: {ds_info.splits['train'].num_examples}개")
print(f"테스트 데이터: {ds_info.splits['test'].num_examples}개")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/rock_paper_scissors/incomplete.ZX4QWD_3.0.0/rock_paper_scissors-train.tfre…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/rock_paper_scissors/incomplete.ZX4QWD_3.0.0/rock_paper_scissors-test.tfrec…

Dataset rock_paper_scissors downloaded and prepared to /root/tensorflow_datasets/rock_paper_scissors/3.0.0. Subsequent calls will reuse this data.
훈련 데이터: 2520개
테스트 데이터: 372개


In [ ]:
# 데이터 전처리 함수
def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # 정규화
    image = tf.image.resize(image, [150, 150])  # 150x150으로 리사이즈
    return image, label

In [ ]:
# 데이터 증강 (Data Augmentation)
def augment_image(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    return image, label

In [ ]:
# 데이터 전처리 및 배치 설정
BATCH_SIZE = 32
BUFFER_SIZE = 1000

In [ ]:
# 훈련 데이터 전처리 (데이터 증강 포함)
train_dataset = ds_train.map(preprocess_image).map(augment_image)
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 테스트 데이터 전처리
test_dataset = ds_test.map(preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# CNN 모델 구성
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dropout(0.5),
    Dense(512, activation='relu'),
    Dense(3, activation='softmax')  # 3개 클래스 (rock, paper, scissors)
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# 모델 컴파일
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# 모델 구조 확인
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     3,211,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,454,147 (13.18 MB)

 Trainable params: 3,454,147 (13.18 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 콜백 설정
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True)
]

In [ ]:
# 모델 훈련
EPOCHS = 20

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4866 - loss: 0.9794

79/79 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.4891 - loss: 0.9754 - val_accuracy: 0.7070 - val_loss: 0.8334
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9653 - loss: 0.0953

79/79 ━━━━━━━━━━━━━━━━━━━━ 126s 2s/step - accuracy: 0.9655 - loss: 0.0951 - val_accuracy: 0.7930 - val_loss: 0.8963
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9926 - loss: 0.0238

79/79 ━━━━━━━━━━━━━━━━━━━━ 128s 2s/step - accuracy: 0.9927 - loss: 0.0238 - val_accuracy: 0.8683 - val_loss: 0.7654
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 140s 2s/step - accuracy: 0.9980 - loss: 0.0066 - val_accuracy: 0.8387 - val_loss: 0.9098
Epoch 5/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.9864 - loss: 0.0297 - val_accuracy: 0.8629 - val_loss: 0.8179
Epoch 6/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9997 - loss: 0.0024

79/79 ━━━━━━━━━━━━━━━━━━━━ 130s 2s/step - accuracy: 0.9997 - loss: 0.0024 - val_accuracy: 0.8898 - val_loss: 0.9979
Epoch 7/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 145s 2s/step - accuracy: 0.9996 - loss: 0.0021 - val_accuracy: 0.8817 - val_loss: 0.8404
Epoch 8/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.9952 - loss: 0.0118 - val_accuracy: 0.8306 - val_loss: 1.4951
Epoch 9/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 145s 2s/step - accuracy: 0.9910 - loss: 0.0383 - val_accuracy: 0.8710 - val_loss: 1.0788
Epoch 10/20
31/79 ━━━━━━━━━━━━━━━━━━━━ 1:14 2s/step - accuracy: 0.9795 - loss: 0.0627

In [ ]:
# 훈련 결과 시각화
plt.figure(figsize=(12, 4))

In [ ]:
# 정확도 그래프
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

In [ ]:
# 손실 그래프
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 최종 평가
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)
print(f"\n최종 테스트 정확도: {test_accuracy:.4f}")
print(f"최종 테스트 손실: {test_loss:.4f}")

In [ ]:
# 예측 함수
def predict_and_show(model, dataset, num_samples=9):
    class_names = ['rock', 'paper', 'scissors']

    plt.figure(figsize=(12, 12))

    # 샘플 이미지 가져오기
    images, labels = next(iter(dataset.take(1)))
    predictions = model.predict(images)

    for i in range(min(num_samples, len(images))):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i])

        predicted_class = np.argmax(predictions[i])
        true_class = labels[i].numpy()

        plt.title(f'real : {class_names[true_class]}\n'
                 f'prediction : {class_names[predicted_class]}\n'
                 f'confidence : {np.max(predictions[i]):.2f}')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# 예측 결과 확인
predict_and_show(model, test_dataset)

In [ ]:
# 모델 저장
model.save('rock_paper_scissors_model.h5')
print("모델이 저장되었습니다!")
